# DeepAgents 05 · 解释器、PTC 与异步子代理

本课把 DeepAgents 官方文档里**两件「把活儿交给代码去做」的事**并在一起讲，它们其实是
`CodeInterpreterMiddleware` 这个中间件的几种用法，外加一个独立的 `AsyncSubAgent`：

| 概念 | 是什么 | 代码形态 |
|---|---|---|
| 解释器 eval | 模型写 JavaScript，在 QuickJS 沙箱里跑，只把结果带回 | `CodeInterpreterMiddleware()` → `eval` 工具 |
| PTC | 把白名单工具以 `tools.camelCase()` 暴露进沙箱，代码可循环调工具 | `CodeInterpreterMiddleware(ptc=["lookup_price"])` |
| 动态子代理 | 沙箱里有个 `task()` 全局函数，代码可循环/并行扇出子代理 | `CodeInterpreterMiddleware(subagents=True)` |
| 异步子代理 | 主管把任务丢给一个 Agent Protocol 服务端，立刻拿 task_id，不阻塞 | `AsyncSubAgent(url=..., graph_id=...)` |

> **本 notebook 由 `Agent/_py_source/03_deepagents/` 下 2 个脚本合并而成**：
> `18_解释器与PTC_官方补充.py`（解释器 / PTC / 动态子代理 → 第 1 节）、
> `19_异步子代理_官方补充.py`（异步子代理 → 第 2 节）。

**官方文档**
- 解释器：<https://docs.langchain.com/oss/python/deepagents/interpreters>
- 异步子代理：<https://docs.langchain.com/oss/python/deepagents/async-subagents>

## 运行条件

| 项 | 说明 |
|---|---|
| 🔴 运行档位 | **需外部服务** —— 第 2 节要自起一个 `langgraph dev`（Agent Protocol 服务端，端口 2024） |
| 依赖 | `deepagents[quickjs]`（提供解释器）、`langgraph-cli[inmem]`（提供 `langgraph dev`）；venv 已装 |
| 密钥 | `settings.api_key` / `base_url` / `model_name`（已配置） |
| 前置服务 | 无（第 2 节的服务在 notebook 内自己起、末尾自己关） |
| 预计耗时 | 约 3~6 分钟（多轮真实模型调用 + 服务启动） |

> 两节都要真实调用大模型；第 2 节还会在本机起一个端口 2024 的本地服务，并在最后一个 cell 关掉。

## 本节地图

```mermaid
graph TD
A["CodeInterpreterMiddleware<br/>给 agent 加 eval 工具"] --> B["eval 基础：模型写 JS 算数"]
A --> C["沙箱边界：无文件/网络/时钟"]
A --> D["PTC：ptc 白名单 → tools.camelCase"]
A --> E["动态子代理：task() 扇出"]
F["AsyncSubAgent<br/>连 Agent Protocol 服务端"] --> G["start / check / list 五工具"]
G --> H["本地 langgraph dev<br/>端口 2024（自起自关）"]
```

这张图等价于下表（裸 JupyterLab 不渲染 mermaid，看表即可）：

| 能力 | 触发方式 | 谁在「写代码」 | 中间结果进上下文吗 |
|---|---|---|---|
| eval 基础 | 模型调 `eval` 工具 | 模型写 JavaScript | 否，只回 `<stdout>` + 最后表达式值 |
| PTC | `ptc=[...]` 白名单 | 模型写循环，代码里 `tools.xxx()` | 否，只在沙箱内往返 |
| 动态子代理 | `subagents=True` | 模型写循环，代码里 `task()` | 否，只回汇总结论 |
| 异步子代理 | `AsyncSubAgent` | 模型调 start/check/list 工具 | 否，主管拿到 task_id 即返回 |

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

> 本课还会用到 `NB_DIR` / `WORKDIR` 两个变量：第 2 节要把现场生成的
> `bg_graph.py` + `langgraph.json` 写到 `WORKDIR` 下的专属子目录，跑完再删。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

## 前置条件自检

本课是三样东西的合集：真实大模型 + 解释器依赖（quickjs）+ `langgraph` CLI。
这一格把它们各查一遍，缺了就打印中文提示，并让后面相关的小节跳过。

In [ ]:
# ===== 前置条件自检（🔴 需外部服务：第 2 节要自起 langgraph dev） =====
import os
import sys
from pathlib import Path

_hints = []

# ① 模型密钥：两节都要真实调用大模型
try:
    from config import settings
    if not getattr(settings, "api_key", ""):
        _hints.append("缺少 settings.api_key（.env 未配置），模型调用会失败")
    READY_MODEL = bool(getattr(settings, "api_key", ""))
except Exception as _e:  # noqa: BLE001
    READY_MODEL = False
    _hints.append(f"无法导入 config.settings：{type(_e).__name__}")

# ② 解释器依赖：deepagents[quickjs]（提供 CodeInterpreterMiddleware）
try:
    from langchain_quickjs import CodeInterpreterMiddleware as _CIM
    READY_INTERPRETER = _CIM is not None
except ImportError:
    READY_INTERPRETER = False
    _hints.append('缺少解释器依赖：请先执行 uv add "deepagents[quickjs]"')

# ③ langgraph CLI：第 2 节的 Agent Protocol 服务端由它提供
_cli = Path(sys.executable).parent / ("langgraph.exe" if os.name == "nt" else "langgraph")
READY_LANGGRAPH = _cli.exists()
if not READY_LANGGRAPH:
    _hints.append('缺少 langgraph CLI：请先执行 uv add "langgraph-cli[inmem]"')

if _hints:
    print("[跳过] 前置条件未就绪：")
    for _h in _hints:
        print("  -", _h)
else:
    print("前置条件自检通过 ✔（模型密钥 / 解释器 / langgraph CLI 均已就绪）")

## 1. 解释器与 PTC（来自 `18_解释器与PTC_官方补充.py`）

`CodeInterpreterMiddleware` 给 agent 加一个 **`eval` 工具**：模型自己写 JavaScript、
调 eval 执行，你不需要直接调解释器。代码跑在 **QuickJS 沙箱**里，默认**没有**
文件系统、网络、shell、包管理器、时钟；只有 `console.log/warn/error` 会被捕获，
并返回最后一个表达式的值。

沙箱只有两条「桥」能通到外面：

- **PTC（程序化工具调用）**：把白名单工具以 `tools.xxx()` 暴露进解释器（默认关闭）；
- **动态子代理**：解释器里有 `task({description, subagentType})` 全局函数（有子代理时默认开启）。

它值得学的原因：多步任务的中间结果**往往只是下一步的输入**。传统做法是
「模型调一次工具 → 等结果 → 再调下一次」，每个中间值都要进上下文；有了解释器，
模型可以写**循环/分支/重试/并行**，在沙箱里把中间结果过滤聚合掉，**只有最终结果回到模型**。

### 1.0 公共部分：依赖、模型与两个观察函数

先装好依赖、建好模型，再定义两个「观察 agent 动作」的小函数：`tool_sequence`
看这一轮 agent 调了哪些工具，`eval_outputs` 取出 eval 工具的返回（解释器的 stdout /
最后表达式值）。后面的每个 Demo 都用它们打印结果。

In [ ]:
import time

from langchain.chat_models import init_chat_model
from langchain.tools import tool

from deepagents import SubAgent, create_deep_agent

# 可选依赖：解释器来自 `deepagents[quickjs]` 这个 extra。按仓库惯例兜住 ImportError，
# 缺包时打印中文提示（而不是 import 就崩栈）。
try:
    from langchain_quickjs import CodeInterpreterMiddleware
except ImportError:  # pragma: no cover - 缺依赖时走降级分支
    CodeInterpreterMiddleware = None  # type: ignore[assignment]

from config import settings

MISSING_HINT = (
    "缺少解释器依赖：请先执行  uv add \"deepagents[quickjs]\"\n"
    "（实测会装上 langchain-quickjs + quickjs-rs + wasmtime 三个包）"
)

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,          # grok-4.6
    api_key=settings.api_key,
    base_url=settings.base_url,
    max_retries=0,
    # 深度智能体的系统提示长、单轮推理量大，网关繁忙时容易超时；给足 4 分钟
    timeout=240,
)


def tool_sequence(result: dict) -> list[str]:
    """把一轮对话里的工具调用名按顺序列出（观察 agent 的"动作"）。"""
    return [
        call["name"]
        for message in result["messages"]
        for call in (getattr(message, "tool_calls", None) or [])
    ]


def eval_outputs(result: dict) -> list[str]:
    """取出 eval 工具的返回（解释器的 stdout / 最后表达式值）。"""
    return [
        str(message.content)
        for message in result["messages"]
        if message.type == "tool" and getattr(message, "name", "") == "eval"
    ]

### 1.1 Demo 1：eval 基础 —— 让模型用代码算，而不是「心算」

中间件构造参数（实测签名）：`memory_limit=64MiB` / `timeout=5.0` 秒 /
`max_ptc_calls=256` / `tool_name="eval"` / `max_result_chars=4000` /
`capture_console=True` / `subagents=True` / `ptc=None` /
`mode=None`（即默认 `"thread"`：解释器状态跨 eval、跨轮次保留）。

下面让模型对一批订单金额做「求和 / 平均 / 最大值 / 计数」，并要求它**写 JavaScript**
用 eval 来算——看它怎么把中间计算留在沙箱里、只把结果带回来。

In [ ]:
def demo_1_eval_basics() -> None:
    print("=" * 70)
    print("Demo 1：eval 基础 —— 模型写 JS，沙箱算，只把结果带回来")
    print("=" * 70)

    agent = create_deep_agent(
        model=llm,
        middleware=[CodeInterpreterMiddleware()],
    )
    data = "订单金额：128、37、512、89、204、76、933、15"
    question = (
        f"这是一批数据：{data}。用 JavaScript 计算总和、平均值、最大值，"
        "并找出超过 200 的订单有几个。"
    )
    started = time.time()
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": 20},
    )
    print(f"  工具调用序列：{tool_sequence(result)}")
    for output in eval_outputs(result):
        print(f"  eval 返回给模型的内容：\n    {output[:220].replace(chr(10), chr(10) + '    ')}")
    print(f"  最终回答（{time.time()-started:.1f}s）：{str(result['messages'][-1].content)[:200]}")
    print(
        "  ↑ 注意 eval 的返回结构：`<stdout>…</stdout><result>…</result>` ——\n"
        "    console.log 的内容进 stdout，**最后一个表达式的值**进 result。\n"
        "    模型拿到的只有这段文本，中间变量（数据数组、循环变量）都没进上下文。"
    )

### 1.2 Demo 2：沙箱边界 —— 为什么它敢让模型跑代码

让模型试着在 eval 里读本机的 `C:\Windows\win.ini`。预期它会发现沙箱里根本没有
`fs` / `require`，或者改用深度智能体自带的文件工具（走虚拟路径，读不到宿主机的
`C:\`）——两种表现都证明「沙箱真的隔离了」。

In [ ]:
def demo_2_sandbox_boundary() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：沙箱边界 —— 没有文件系统、网络、时钟")
    print("=" * 70)

    agent = create_deep_agent(
        model=llm,
        middleware=[CodeInterpreterMiddleware()],
    )
    result = agent.invoke(
        {
            "messages": [{
                "role": "user",
                "content": "用 eval 里的 JavaScript 读一下 C:\\Windows\\win.ini 的内容，打印出来。",
            }]
        },
        config={"recursion_limit": 20},
    )
    print(f"  工具调用序列：{tool_sequence(result)}")
    print(f"  最终回答：{str(result['messages'][-1].content)[:220]}")
    called = tool_sequence(result)
    fs_calls = [name for name in called if name in ("read_file", "ls", "glob", "grep")]
    print(f"  ↑ 本次实际调用：{called or '（无）'}")
    print(
        "    常见表现有两种，都算正常：\n"
        "      · 直接改用深度智能体自带的文件工具" + ("（本次就是：%s）" % fs_calls if fs_calls else "") + "\n"
        "        —— 那些工具走**虚拟路径**，读不到宿主机的 C:\\（后端抽象的隔离效果）；\n"
        "      · 先在 eval 里试 fs / require —— 沙箱里根本没有，会拿到 TypeError。\n"
        "    两句话记牢：① eval 沙箱没有文件系统、网络、时钟；\n"
        "    ② 要让 agent 读写文件，走**后端 + 文件工具**（课案 03~09 的后端体系），\n"
        "    而不是把宿主环境暴露给解释器。\n"
        "    （模型对环境/自身的描述（比如「当前环境不是 Windows」）不可信，能力边界以实测为准。）"
    )

### 1.3 Demo 3：PTC —— 让代码循环调用工具（token 效率的关键）

- 传统模式：模型 → 调工具 → 等结果 → 再调 → …（每个中间结果都进上下文）
- PTC 模式：模型 → 写一段循环代码 → 代码在沙箱里连续调用工具 → 只把聚合结果带回来

开关是 `CodeInterpreterMiddleware(ptc=["工具名", ...])`，默认关闭（白名单机制）。
下面注册一个模拟外部接口的 `lookup_price` 工具，让它以 `tools.lookupPrice(...)`
暴露进沙箱，并让模型「写一段 JavaScript 循环」去查五个单价、算总价、找最贵。

In [ ]:
# 传统模式：模型 → 调工具 → 等结果 → 再调 → …（每个中间结果都进上下文）
# PTC 模式：模型 → 写一段循环代码 → 代码在沙箱里连续调用工具 → 只把聚合结果带回来
# 开关：CodeInterpreterMiddleware(ptc=["工具名", ...])，默认关闭（白名单机制）。
PTC_CALLS = {"count": 0}


@tool
def lookup_price(product: str) -> str:
    """查询商品单价（模拟外部接口，每次调用都算一次"网络往返"）。"""
    PTC_CALLS["count"] += 1
    prices = {"钢笔": 12.5, "笔记本": 8.0, "台灯": 79.0, "键盘": 259.0, "鼠标": 89.0}
    return f"{product}={prices.get(product, '未知')}"


def demo_3_ptc() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：PTC —— 模型写循环调用工具，只把聚合结果带回来")
    print("=" * 70)

    agent = create_deep_agent(
        model=llm,
        tools=[lookup_price],
        middleware=[
            # ptc 白名单：把 lookup_price 以 tools.lookupPrice(...) 暴露进解释器
            CodeInterpreterMiddleware(ptc=["lookup_price"]),
        ],
    )
    PTC_CALLS["count"] = 0
    question = (
        "请查出这五样东西的单价：钢笔、笔记本、台灯、键盘、鼠标，"
        "算出一共多少钱，并告诉我最贵的是哪个。"
        "不要一个个手动查，写一段 JavaScript 循环调用工具来完成。"
    )
    started = time.time()
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": 30},
    )
    print(f"  工具调用序列（模型层面的）：{tool_sequence(result)}")
    print(f"  lookup_price 被调用的总次数：{PTC_CALLS['count']}"
          "（含模型直连工具的次数；若模型按提示走 PTC，这些调用都发生在解释器里）")
    for output in eval_outputs(result):
        print(f"  eval 返回：{output[:200]}")
    print(f"  最终回答（{time.time()-started:.1f}s）：{str(result['messages'][-1].content)[:200]}")
    print(
        "  ↑ 这一段建议逐行读输出：**模型不是一次就成的**。\n"
        "    实测见过几种失败形态（每次跑不一样，但都属同一类问题）：\n"
        "      · `TypeError: not a function` —— 工具名/调用形态猜错了（记住是 camelCase）；\n"
        "      · 结果里出现 `nan` —— 拿到的返回值是字符串，没先解析就参与计算；\n"
        "      · `SyntaxError: redeclaration of 'total'` —— 解释器状态**跨 eval 保留**\n"
        "        （默认 mode=\"thread\"），上一轮的 `const total` 还在，重复声明就报错。\n"
        "    模型会自己看报错、改代码、再试 —— 这正是解释器模式的价值：\n"
        "    **试错发生在沙箱里**，模型上下文只承受「报错一行 + 最终结果」，\n"
        "    而不是把每次工具往返的中间值都塞进去。\n"
        "    另外：状态保留是双刃剑 —— 想每次 eval 都是干净环境就设 `mode=\"call\"`。"
    )

### 1.4 Demo 4：动态子代理 —— 在代码里 `task()` 扇出

官方 dynamic-subagents 说明：有子代理时，解释器里会多一个 `task()` 全局函数，
可以在代码里循环/并行派发子代理，再统一汇总。适用：批量同构工作（逐个文件审查、
批量工单分诊）。下面配一个 `order-checker` 子代理，让模型在 eval 里用 `Promise.all`
并行把三个订单派给它再汇总。

In [ ]:
def demo_4_dynamic_subagents() -> None:
    print("\n" + "=" * 70)
    print("Demo 4：动态子代理 —— 解释器里用 task() 扇出")
    print("=" * 70)

    agent = create_deep_agent(
        model=llm,
        middleware=[
            # subagents 默认 True；这里显式写出来便于阅读
            CodeInterpreterMiddleware(subagents=True),
        ],
        subagents=[
            SubAgent(
                name="order-checker",
                description="检查单个订单是否合规（参数：订单号），返回一句结论",
                system_prompt=(
                    "你是订单合规检查员。用户给你一个订单号，"
                    "你只回一句「订单 X：合规/不合规 + 原因」。"
                ),
                model=llm,
                tools=[],
            )
        ],
    )
    question = (
        "有三个订单需要检查：A-1001、A-1002、A-1003。"
        "请**调用 eval 工具执行**一段 JavaScript：用 Promise.all 并行把三个订单派给 "
        "order-checker 子代理（用 task({description, subagentType})），然后汇总三句结论。"
        "只把代码贴出来不算完成 —— 必须真的跑出结果。"
    )
    started = time.time()
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": 40},
    )
    print(f"  工具调用序列：{tool_sequence(result)}")
    for output in eval_outputs(result):
        print(f"  eval 返回（子代理汇总）：\n    {output[:260].replace(chr(10), chr(10) + '    ')}")
    print(f"  最终回答（{time.time()-started:.1f}s）：{str(result['messages'][-1].content)[:220]}")
    if "eval" in tool_sequence(result):
        print("  ✔ 本次确实在沙箱里跑起来了（工具序列里有 eval）")
    else:
        print(
            "  ⚠️ 本次模型只把代码**写出来**、没调用 eval 执行（工具序列里没有 eval）——\n"
            "     这是模型行为：它更习惯「给答案」而不是「动手跑」。\n"
            "     实践上要么在提示词里把「必须执行」写得更硬，要么用 Rubric 校验（见 16 号文件）。"
        )
    print(
        "  ↑ 与课案 03_deepagents/12 的静态子代理区别：\n"
        "    · 静态：模型逐个用 task 工具派发（派几次由模型决定，每次都进上下文）；\n"
        "    · 动态：**在代码里**循环/并行派发（次数由代码决定，只有汇总结果回上下文）。\n"
        "    官方给的典型场景：逐个文件审查、批量工单分诊、结果再交叉验证。"
    )

### 1.5 执行入口：逐 Demo 兜底

下面这一格就是原 `18` 号文件 `if __name__ == "__main__"` 里的内容（去掉那层包裹，
顶格写出）。它按顺序跑四个 Demo，每个 Demo 单独兜底：真实模型 + 解释器，网关抖动时
打印提示并继续，不让一个 Demo 的失败拖垮整本。

In [ ]:
if CodeInterpreterMiddleware is None:
    print(MISSING_HINT)
    print("[跳过] 解释器依赖未就绪，第 1 节四个 Demo 不执行。")
else:
    for _demo in (demo_1_eval_basics, demo_2_sandbox_boundary, demo_3_ptc, demo_4_dynamic_subagents):
        try:
            _demo()
        except Exception as _exc:  # noqa: BLE001
            print(f"\n  ⚠️ {_demo.__name__} 本次未跑完（网关抖动/超时，非代码问题）：{type(_exc).__name__}")
            print("  重跑一次通常即可。")
    print("\n全部 Demo 执行完毕。")

### 预期输出

```text
======================================================================
Demo 1：eval 基础 —— 模型写 JS，沙箱算，只把结果带回来
======================================================================
  工具调用序列：['eval']
  eval 返回给模型的内容：
    <stdout>...</stdout><result>...</result>
  最终回答（X.Xs）：...
  ↑ 注意 eval 的返回结构：`<stdout>…</stdout><result>…</result>` —— ...

======================================================================
Demo 2：沙箱边界 —— 没有文件系统、网络、时钟
======================================================================
  工具调用序列：[...]
  最终回答：...
  ↑ 本次实际调用：...

======================================================================
Demo 3：PTC —— 模型写循环调用工具，只把聚合结果带回来
======================================================================
  工具调用序列（模型层面的）：['eval']
  lookup_price 被调用的总次数：5（...）
  eval 返回：...
  最终回答（X.Xs）：...

======================================================================
Demo 4：动态子代理 —— 解释器里用 task() 扇出
======================================================================
  工具调用序列：['eval']
  eval 返回（子代理汇总）：...
  最终回答（X.Xs）：...

全部 Demo 执行完毕。
```

> ⚠️ 上面的正文只是**结构示意**：模型措辞、工具调用序列、eval 返回内容、耗时（`X.Xs`）
> 每次都不同，别逐字比对；稳定的是「分隔线 + Demo 标题 + 工具调用序列 + 最终回答」这个骨架。

## 2. 异步子代理（来自 `19_异步子代理_官方补充.py`）

上一节的子代理是**同步**的：主管调 task 工具后**阻塞**等结果。异步子代理正好相反：

| 维度 | 同步子代理（课案 12） | 异步子代理（本文件） |
|---|---|---|
| 执行模型 | 主管**阻塞**等子代理跑完 | 立刻返回 task_id，主管继续对话 |
| 中途干预 | 做不到 | 可以用 update_async_task 追加指令 |
| 取消 | 做不到 | 可以 cancel_async_task |
| 状态 | 无状态 | 有独立线程、跨交互保留状态 |
| 适用 | 必须先拿到结果才能继续 | 长耗时 / 可并行 / 需要边跑边聊的任务 |

官方原文的关键约束：「Async subagents communicate with **any server that implements
the Agent Protocol**.」——也就是说，异步子代理**必须有一个服务端**（子代理跑在那边，
主管通过 SDK 管它）。本课用「自己起服务」的方案（不依赖 LangSmith）：现场生成一个最小
LangGraph 应用，用 `langgraph dev --port 2024` 起本地 Agent Protocol 服务端，主管用
`AsyncSubAgent(url="http://127.0.0.1:2024", graph_id="bg-graph")` 连上去。

### 2.0 公共部分：依赖、模型与后台图源码

这里有两个本机实测踩坑，都在下面代码里处理了：

- **中文 Windows 必须开 UTF-8 模式**：`langgraph dev` 启动时会在
  `langgraph_api/validation.py` 里用默认编码（GBK）读 UTF-8 文件，直接
  `UnicodeDecodeError: 'gbk' codec can't decode byte 0x94`。解法：给子进程设 `PYTHONUTF8=1`。
- **`url=None`（ASGI 传输）在本地跑不通**：ASGI 应用只在部署上下文里存在，本地演示必须显式给 url。

In [ ]:
import sys
import json
import os
import re
import shutil
import subprocess
import tempfile
import time
import urllib.error
import urllib.request
from pathlib import Path

from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

from deepagents import AsyncSubAgent, create_deep_agent

from config import settings

REPO_ROOT = ROOT      # 源文件写 Path(__file__).resolve().parents[2]，notebook 里直接用 bootstrap 算出的 ROOT
PORT = 2024
BASE_URL = f"http://127.0.0.1:{PORT}"
GRAPH_ID = "bg-graph"

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
    max_retries=0,
    # 本机网关在负载高时单次请求可能超过 2 分钟，这里放宽到 4 分钟
    timeout=240,
)

### 2.1 现场生成最小后台图

`BG_GRAPH_SOURCE` 就是现场生成的最小「后台图」：一个只会干活、不啰嗦的单轮 agent。
它会在起服务前被写进临时目录的 `bg_graph.py`，配合 `langgraph.json` 一起交给
`langgraph dev` 托管。注意里面 `sys.path.insert(0, r"{REPO_ROOT}")` 把仓库根插进子进程的
`sys.path`，这样服务端进程里的后台图才能 `from config import settings` 拿到模型配置。

In [ ]:
# 现场生成的最小"后台图"：一个只会干活、不啰嗦的单轮 agent
BG_GRAPH_SOURCE = f'''
"""langgraph dev 托管的后台图（由 19_异步子代理_官方补充.py 现场生成）。"""
import sys

sys.path.insert(0, r"{REPO_ROOT}")      # 让子进程能找到仓库根的 config.py

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

from config import settings

llm = init_chat_model(
    model_provider="openai", model=settings.model_name,
    api_key=settings.api_key, base_url=settings.base_url,
    max_retries=0, timeout=240,
)

graph = create_agent(
    model=llm,
    tools=[],
    system_prompt="你是后台任务执行器。认真完成任务，用一句话汇报结果。",
)
'''

### 2.2 服务端生命周期函数

这组函数负责「起 / 探 / 等 / 关 / 清」一个本地 Agent Protocol 服务端：

- `server_is_up()` 单次探测 `/ok`；
- `wait_for_server()` 轮询等就绪（最多 90 秒）；
- `start_agent_protocol_server()` 现场写 `bg_graph.py` + `langgraph.json`，然后
  `subprocess.Popen` 后台起 `langgraph dev`（**不阻塞内核**），并设 `PYTHONUTF8=1`；
- `stop_agent_protocol_server()` 关掉**本文件起的**服务（Windows 用 `taskkill /F /T` 连子树收）；
- `remove_temp_dir()` 删临时目录，**重试 + 容忍失败**（刚被 taskkill 的句柄不会立刻释放）。

In [ ]:
def server_is_up() -> bool:
    """单次探测：服务端现在是否已经就绪（不等待）。"""
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    try:
        with opener.open(f"{BASE_URL}/ok", timeout=2) as response:
            return response.status == 200
    except (urllib.error.URLError, OSError):
        return False


def wait_for_server(timeout: int = 90) -> bool:
    """轮询 /ok，确认 Agent Protocol 服务端已就绪（最多等 timeout 秒）。"""
    deadline = time.time() + timeout
    while time.time() < deadline:
        if server_is_up():
            return True
        time.sleep(1.5)
    return False


def start_agent_protocol_server(workdir: Path):
    """在临时目录里生成最小应用并起 langgraph dev，返回 Popen 对象。

    返回值语义：**进程对象 = 本文件起的（结束时由本文件关）；None = 复用已有服务**
    （上一次运行留下了孤儿进程时会出现这种情况 —— 复用比重复起一个必然失败的更稳，
    而且绝不误杀别人的进程）。
    """
    if server_is_up():
        print(f"  检测到 {BASE_URL} 已有服务在运行 → 复用它（本次不新起进程）")
        return None

    (workdir / "bg_graph.py").write_text(BG_GRAPH_SOURCE, encoding="utf-8")
    (workdir / "langgraph.json").write_text(
        json.dumps({"dependencies": ["."], "graphs": {GRAPH_ID: "./bg_graph.py:graph"}}, indent=2),
        encoding="utf-8",
    )

    # 关键：PYTHONUTF8=1（中文 Windows 上否则会因为 GBK 读 UTF-8 文件而启动失败）
    env = os.environ.copy()
    env["PYTHONUTF8"] = "1"
    env["LANGSMITH_TRACING"] = "false"
    env["NO_PROXY"] = "127.0.0.1,localhost"

    cli = Path(sys.executable).parent / ("langgraph.exe" if os.name == "nt" else "langgraph")
    if not cli.exists():
        raise FileNotFoundError(
            f"找不到 langgraph CLI：{cli}\n"
            '请先安装：uv add "langgraph-cli[inmem]"（本文件的服务端由它提供）'
        )
    process = subprocess.Popen(
        [str(cli), "dev", "--port", str(PORT), "--host", "127.0.0.1", "--no-browser"],
        cwd=str(workdir),
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        # 单独进程组，便于结束时连同子进程一起收掉
        creationflags=subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0,
    )
    return process


def stop_agent_protocol_server(process) -> None:
    """关掉**本文件起的**服务端（Windows 上连同子进程树一起收）。

    process 为 None（复用已有服务）时不动作 —— 别人起的服务不该由我们关掉。
    """
    if process is None:
        print("  （本次复用的是已有服务，不做关闭动作）")
        return
    if process.poll() is not None:
        return
    if os.name == "nt":
        subprocess.run(["taskkill", "/F", "/T", "/PID", str(process.pid)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    else:
        # POSIX：langgraph dev 会 fork uvicorn，只 terminate 父进程会留下子进程占端口
        try:
            import os as _os
            import signal as _signal
            _os.killpg(_os.getpgid(process.pid), _signal.SIGTERM)
        except Exception:  # noqa: BLE001
            process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()


def remove_temp_dir(path: Path) -> None:
    """删掉临时目录；在 Windows 上删不掉**也不算失败**。

    ⚠️ 实测踩坑：`langgraph dev` 的进程树刚被 taskkill 掉时，Windows **不会立刻**
    释放它占用的文件句柄。紧接着 `shutil.rmtree` 就会抛
    `PermissionError: [WinError 32] 另一个程序正在使用此文件，进程无法访问。`
    —— 整份演示明明全部跑完了，却崩在最后一行清理上。
    所以这里：先小步重试（等句柄释放），最后一步容忍失败 ——
    清理不掉只是留下一个临时目录，绝不该让演示报错。
    """
    for _ in range(5):
        try:
            shutil.rmtree(path)
            return
        except FileNotFoundError:
            return
        except OSError:
            time.sleep(1.0)      # 给 Windows 一点时间释放进程持有的句柄
    shutil.rmtree(path, ignore_errors=True)
    if path.exists():
        print(f"  （临时目录未能删除，可手动清理：{path}）")

### 2.3 主管与两个观察函数

`build_supervisor()` 配一个异步子代理 `bg-worker`。实测要点：**异步任务的跟踪挂在会话
线程上**——想让 check/list 看得到任务，必须用**同一个 agent 实例 + 同一个 thread_id**
（配 checkpointer + config 传 thread_id），否则只会得到「No tracked task found」。

In [ ]:
def tool_sequence(result: dict) -> list[str]:
    return [
        call["name"]
        for message in result["messages"]
        for call in (getattr(message, "tool_calls", None) or [])
    ]


def tool_outputs(result: dict) -> list[str]:
    return [str(m.content) for m in result["messages"] if m.type == "tool"]


def build_supervisor(checkpointer=None):
    """主管 agent：配一个异步子代理（必须给 url，否则本地跑不通 —— 见文件头 B 条）。

    实测要点：**异步任务的跟踪挂在会话线程上** —— 想让 check/list 看得到任务，
    必须用**同一个 agent 实例 + 同一个 thread_id**（配 checkpointer + config 传 thread_id）。
    否则新建 agent 再查，只会得到「No tracked task found」。
    """
    return create_deep_agent(
        model=llm,
        subagents=[
            AsyncSubAgent(
                name="bg-worker",
                description="后台任务执行器：适合把长耗时/可并行的活儿丢过去慢慢做",
                graph_id=GRAPH_ID,                 # 必须与 langgraph.json 里注册的图名一致
                url=BASE_URL,                      # ← 本地 Agent Protocol 服务地址
            )
        ],
        checkpointer=checkpointer,
    )

### 2.4 Demo 1：启动后台任务 —— 主管不阻塞，立刻拿到 task_id

让主管把「统计 1 到 20 的和」这件事丢到后台做，拿到 task_id 就回复。

In [ ]:
def demo_1_launch() -> None:
    print("=" * 70)
    print("Demo 1：start_async_task —— 丢出去就返回，不阻塞主管")
    print("=" * 70)

    agent = build_supervisor()
    started = time.time()
    result = agent.invoke(
        {
            "messages": [{
                "role": "user",
                "content": "把「统计 1 到 20 的和，并说明计算过程」这件事放到后台去做，拿到任务 ID 就告诉我。",
            }]
        },
        config={"recursion_limit": 20},
    )
    elapsed = time.time() - started
    print(f"  工具调用序列：{tool_sequence(result)}")
    for output in tool_outputs(result):
        print(f"  工具返回：{output[:150]}")
    print(f"  主管回答（{elapsed:.1f}s）：{str(result['messages'][-1].content)[:160]}")
    print(
        "  ↑ 注意耗时构成：主管**发完就返回**了（不等后台跑完）——\n"
        "    这正是异步子代理与课案 03_deepagents/12 的同步子代理的根本区别。\n"
        "    同步版会一直阻塞到子代理给出结果；异步版立刻拿到 task_id，之后随时来查。"
    )

### 2.5 Demo 2：查状态 check + 列任务 list

用正则从工具返回里抽出 `task_id`，等 8 秒再查状态、列任务。五个工具的分工（官方表格）：

- `start_async_task` 启动，立刻返回 task_id
- `check_async_task` 查状态与结果（running / success / error …）
- `list_async_tasks` 列出所有任务
- `update_async_task` 给正在跑的任务追加指令（本课未演示）
- `cancel_async_task` 取消任务（本课未演示）

In [ ]:
def demo_2_check_and_list() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：check_async_task / list_async_tasks —— 事后查进度")
    print("=" * 70)

    # 关键：同一个 agent 实例 + 同一个 thread_id（配 checkpointer），任务才"看得见"
    agent = build_supervisor(checkpointer=InMemorySaver())
    config = {"configurable": {"thread_id": "async-demo-thread"}, "recursion_limit": 20}

    launched = agent.invoke(
        {"messages": [{"role": "user", "content": "后台算一下 100 的阶乘有多少位数字，拿到任务 ID 即可。"}]},
        config,
    )
    # 用正则抽 task_id：模型同一轮里可能既调 start 又调 check（后者的返回是 JSON 形状的
    # "task_id": "..."），用 split("task_id:") 会切出带引号/空串的垃圾值 —— 本文件踩过。
    task_id = ""
    for output in tool_outputs(launched):
        match = re.search(r"task_id[\"']?\s*[:：]\s*[\"']?([0-9a-fA-F-]{8,})", output)
        if match:
            task_id = match.group(1)
            break
    if not task_id:
        print("  ⚠️ 没能从工具返回里解析出 task_id（返回内容见上），后续 check 会查不到任务。")
    print(f"  已启动任务：{task_id or '（没拿到 ID）'}")

    print("  等 8 秒再查状态（给后台一点时间）…")
    time.sleep(8)

    check = agent.invoke(
        {"messages": [{"role": "user", "content": f"查一下任务 {task_id} 现在的状态和结果。"}]},
        config,     # ← 同一个 thread_id：任务跟踪信息在这里
    )
    print(f"  查询用的工具：{tool_sequence(check)}")
    for output in tool_outputs(check):
        print(f"  工具返回：{output[:220]}")
    print(f"  主管回答：{str(check['messages'][-1].content)[:180]}")

    listing = agent.invoke(
        {"messages": [{"role": "user", "content": "把所有后台任务列出来给我。"}]},
        config,
    )
    print(f"\n  列表用的工具：{tool_sequence(listing)}")
    print(f"  主管回答：{str(listing['messages'][-1].content)[:220]}")
    print(
        "  ↑ 五个工具的分工（官方表格）：\n"
        "    start_async_task  启动，立刻返回 task_id\n"
        "    check_async_task  查状态与结果（running / success / error …）\n"
        "    list_async_tasks  列出所有任务\n"
        "    update_async_task 给正在跑的任务追加指令（本文件未演示）\n"
        "    cancel_async_task 取消任务（本文件未演示）\n"
        "    中间件会自动处理线程创建、运行管理与状态持久化 —— 你只写工具调用即可。\n"
        "    但**跟踪信息挂在会话线程上**（本文件实测踩过）：换 agent 实例或换 thread_id\n"
        "    再查会得到“No tracked task found”，所以一定要配 checkpointer + 固定 thread_id。"
    )

### 2.6 起服务 + 演示 + 关停（一镜到底）

这一格把「起服务 → 等就绪 → 跑 Demo 1/2 → 关服务 → 删临时目录」串成一条链，
用 `try/finally` 保证**无论 Demo 是否跑通，服务都会被关掉**。应用文件写到
`WORKDIR / "agent_protocol"`（同章 `tmp_nb_work` 是共享容器，必须套一层专属子目录）。

In [ ]:
workdir = WORKDIR / "agent_protocol"
workdir.mkdir(exist_ok=True)
try:
    # 先判断端口上是否已有服务：有就复用（那时**不会**生成应用文件，也不该打印"已生成"）
    if server_is_up():
        print(f"检测到 {BASE_URL} 已有服务 → 复用（本次不生成应用、不新起进程）")
        print("  ⚠️ 注意：复用的服务加载的是**它自己启动时**那份图定义；")
        print("     若你刚改过 BG_GRAPH_SOURCE，请先杀掉旧服务（或换端口）再跑，否则验的是旧代码。")
        process = None
    else:
        print(f"生成最小 Agent Protocol 应用（临时目录，跑完即删）：{workdir}")
        process = start_agent_protocol_server(workdir)
    try:
        if process is not None:
            print(f"启动服务 {BASE_URL}（langgraph dev，PYTHONUTF8=1）…")
        if not wait_for_server():
            print(
                "服务未能在 90 秒内就绪。手动复现时**别照抄上面的临时目录**"
                "（它在本文件结束时就删了），自己建一个目录放同样的两个文件即可：\n"
                "  bg_graph.py（内容见本文件头部的模板）+ langgraph.json\n"
                "  $env:PYTHONUTF8='1'; langgraph dev --port 2024\n"
                "然后重跑本文件（它会复用已就绪的服务）。"
            )
            raise SystemExit(1)
        print("服务已就绪 ✔\n")
        try:
            demo_1_launch()
        except Exception as _exc:  # noqa: BLE001
            print(f"  ⚠️ demo_1_launch 本次未跑完（网关抖动/超时，非代码问题）：{type(_exc).__name__}")
            print("  重跑一次通常即可。")
        try:
            demo_2_check_and_list()
        except Exception as _exc:  # noqa: BLE001
            print(f"  ⚠️ demo_2_check_and_list 本次未跑完（网关抖动/超时，非代码问题）：{type(_exc).__name__}")
            print("  重跑一次通常即可。")
    finally:
        stop_agent_protocol_server(process)
        if process is not None:
            print("\n本地 Agent Protocol 服务已关闭（含子进程树）。")
finally:
    # 关服务之后再删目录：服务收不干净时句柄还占着，删不掉也无所谓
    remove_temp_dir(workdir)
print("全部 Demo 执行完毕。")

### 预期输出

第 2 节的输出分三段：先打印「服务已就绪 ✔」，然后 Demo 1/2 各自的工具序列与主管回答，
最后打印「本地 Agent Protocol 服务已关闭」与「全部 Demo 执行完毕」。

```text
生成最小 Agent Protocol 应用（临时目录，跑完即删）：...
启动服务 http://127.0.0.1:2024（langgraph dev，PYTHONUTF8=1）…
服务已就绪 ✔

======================================================================
Demo 1：start_async_task —— 丢出去就返回，不阻塞主管
======================================================================
  工具调用序列：['start_async_task']
  工具返回：Launched async subagent. task_id: ...
  主管回答（X.Xs）：...

======================================================================
Demo 2：check_async_task / list_async_tasks —— 事后查进度
======================================================================
  已启动任务：...-...-...
  等 8 秒再查状态（给后台一点时间）…
  查询用的工具：['check_async_task']
  工具返回：...
  主管回答：...
  列表用的工具：['list_async_tasks']
  主管回答：...

本地 Agent Protocol 服务已关闭（含子进程树）。
全部 Demo 执行完毕。
```

> ⚠️ 模型措辞、task_id（随机 UUID）、工具返回内容与耗时（`X.Xs`）每次都不同，别逐字比对；
> 稳定的是「服务已就绪 ✔ / Demo 标题 / 已启动任务 / 服务已关闭 / 全部 Demo 执行完毕」这些骨架。

## 小结

- **解释器 eval** 给 agent 加 `eval` 工具，模型写 JavaScript 在 QuickJS 沙箱里跑，
  返回 `<stdout>…</stdout><result>…</result>`，中间变量不进上下文；
- **沙箱默认无文件系统 / 网络 / shell / 包管理器 / 时钟**，只有 console.log 被捕获；
- **PTC** 用 `ptc=[...]` 白名单把工具以 `tools.camelCase()` 暴露进沙箱（默认关闭），
  让代码能循环调工具、只回聚合结果 —— 省 token、少往返；
- **动态子代理** 靠解释器里的 `task({description, subagentType})` 在代码里循环/并行扇出；
- **异步子代理** 必须连一个 Agent Protocol 服务端；本地可用 `langgraph dev --port 2024`
  自起，主管用 `AsyncSubAgent(url=..., graph_id=...)` 连上，五个工具 start / check / update / cancel / list；
- 两者共享同一个动机：**把循环与中间结果的过滤丢进沙箱/服务端，只把结论带回模型上下文**。

## 常见坑

1. **解释器是 JavaScript（QuickJS），不是 Python** —— 提示词里要写清「用 JavaScript」，
   否则模型可能写 Python 然后报语法错。
2. **PTC 默认关闭**，必须显式给 `ptc=[...]` 白名单；工具名在解释器里变成 camelCase。
3. **沙箱没有时钟**：需要当前时间要自己通过工具/上下文提供（`new Date()` 不可靠）。
4. **解释器状态默认跨轮次保留（`mode="thread"`）**：上一轮的 `const total` 还在，
   重复声明会报 `SyntaxError: redeclaration`；要干净环境就用 `mode="call"`。
5. **中文 Windows 上 `langgraph dev` 必须 `PYTHONUTF8=1`**：否则 `langgraph_api/validation.py`
   用 GBK 读 UTF-8 文件直接 `UnicodeDecodeError`。
6. **本地不要用 `url=None`（ASGI）**：ASGI 应用只在部署上下文里存在，本地演示必须显式给 url。
7. **`graph_id` 必须与 langgraph.json 里注册的名字完全一致**，否则启动任务就失败。
8. **异步任务跟踪挂在会话线程上**：要配 checkpointer + 固定 thread_id，换 agent 实例或
   thread_id 再查会得到「No tracked task found」。
9. **关服务要连子进程树一起收**（Windows 用 `taskkill /F /T`）：`langgraph dev` 会 fork 出
   uvicorn 子进程，只 terminate 父进程会留下孤儿端口占用。

## 官方链接

- 解释器（Interpreters + PTC + 持久化）：<https://docs.langchain.com/oss/python/deepagents/interpreters>
- 动态子代理（dynamic subagents）：<https://docs.langchain.com/oss/python/deepagents/dynamic-subagents>
- 异步子代理（async subagents）：<https://docs.langchain.com/oss/python/deepagents/async-subagents>